# Setup

In [0]:
from pyspark.sql import functions as F, SparkSession
spark = SparkSession.builder.appName("Gold Layer").getOrCreate()

In [0]:
transactions=spark.table('databricks_fundamentals.silver.transactions')
cards=spark.table('databricks_fundamentals.silver.cards')
users=spark.table('databricks_fundamentals.silver.users')

In [0]:
fraud_pie_chart=transactions.withColumn('Fraud_Label', F.when(F.col('Fraud_Label')==True,'Fraud').when(F.col('Fraud_Label')==False,'Not Fraud').otherwise('Unknown')).groupBy("Fraud_Label").agg(F.count('Fraud_Label').alias('Fraud_Count'))
fraud_pie_chart.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.fraud_pie_chart')
display(fraud_pie_chart)

Fraud_Label,Fraud_Count
Unknown,4390952
Fraud,13332
Not Fraud,8901631


In [0]:
transactions=transactions.filter(F.col('Fraud_Label').isNotNull())

# Analysis

### Question 1: 
Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
fraud_days=transactions.filter(F.col("Fraud_Label")==True).withColumn('dayOfWeekInNumbers',F.dayofweek(F.col('date'))).withColumn('Day_of_the_Week', F.date_format('date','EEEE')).groupBy('dayOfWeekInNumbers','Day_of_the_Week').agg(F.count('Fraud_Label').alias('Fraudulent_Transactions')).orderBy('dayOfWeekInNumbers').drop('dayOfWeekInNumbers')
fraud_days.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.fraud_days')
display(fraud_days)

Day_of_the_Week,Fraudulent_Transactions
Sunday,2646
Monday,1747
Tuesday,2037
Wednesday,1102
Thursday,2082
Friday,2284
Saturday,1434


### Question 2:
What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

In [0]:
year_month_df=transactions.withColumn('Year-Month',F.date_format(F.col('date'),'yyyy-MM'))
total_transactions=year_month_df.groupBy('Year-Month').agg(F.count('id').alias('total_transactions')).orderBy('Year-Month')
fraud_transactions=year_month_df.filter(F.col('Fraud_Label')==True).groupBy('Year-Month').agg(F.count('id').alias('fraud_transactions')).orderBy('Year-Month')
Monthly_rate=total_transactions.join(fraud_transactions, on='Year-Month', how='left').withColumn('fraud_transactions',F.when(F.col('fraud_transactions').isNull(),0).otherwise(F.col('fraud_transactions'))).withColumn('Monthly_Rate',F.col('fraud_transactions')/F.col('total_transactions')).orderBy("Year-Month")
Monthly_rate.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.Monthly_rate')
display(Monthly_rate)

Year-Month,total_transactions,fraud_transactions,Monthly_Rate
2010-01,68044,107,0.0015725119040620775
2010-02,62816,259,0.004123153336729496
2010-03,69202,261,0.0037715672957428976
2010-04,66729,237,0.003551679179966731
2010-05,70210,274,0.0039025779803446804
2010-06,68694,182,0.0026494308090954087
2010-07,70891,244,0.0034419037677561326
2010-08,72043,229,0.003178657190844357
2010-09,69659,193,0.002770639831177594
2010-10,71342,224,0.0031398054441983685


In [0]:
lastMonth=Monthly_rate.agg(F.max(F.col('Year-Month')))
lastMonthFiltered=Monthly_rate.filter(F.col('Year-Month')==lastMonth.first()[0])
display(lastMonthFiltered)




Year-Month,total_transactions,fraud_transactions,Monthly_Rate
2019-10,78379,177,0.002258257951747279


### Question 3
Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
users_fraud_count=transactions.join(users,users.id==transactions.client_id,how='inner').filter(F.col('Fraud_Label')==True).groupBy('client_id').agg(F.count('transactions.id').alias('user_fraudlant_transactions_count')).orderBy('user_fraudlant_transactions_count', ascending=False)
users_fraud_count.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.users_fraud_count')
display(users_fraud_count)

client_id,user_fraudlant_transactions_count
1102,58
209,52
27,45
155,44
1128,43
989,42
1741,42
1851,42
1649,41
1416,39


### Question 4: 
Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:

user_spike_rise=transactions.withColumn('year-week',F.concat_ws("-",F.year(F.col('date')),F.weekofyear(F.col('date')))).filter(F.col('amount')>0).groupBy('client_id','year-week').agg(F.avg('amount').alias('weekly_avg'),F.max('amount').alias('largest_transaction')).withColumn("spike_ratio", F.col("largest_transaction") / F.col("weekly_avg")).filter(F.col("spike_ratio") >= 2).orderBy('client_id').dropDuplicates()
user_spike_rise.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.user_spike_rise')
display(user_spike_rise)     

client_id,year-week,weekly_avg,largest_transaction,spike_ratio
0,2016-11,51.161111,142.53,2.7859050988943536
0,2012-28,50.048421,118.18,2.3613132570156409
0,2019-9,32.306667,111.72,3.4581097455828545
0,2013-27,96.845263,609.69,6.2955066785248959
0,2017-24,58.302500,294.52,5.0515844089018481
0,2016-1,42.596875,130.61,3.0661873670310322
0,2015-6,35.826429,160.00,4.4659767793212100
0,2017-27,39.442500,83.94,2.1281612473854345
0,2011-41,53.633333,165.20,3.0801740402745434
0,2018-35,58.987000,316.15,5.3596555173173750


### Question 5:
Which merchant categories exhibit the highest fraud rate?

In [0]:
merchant_category_total=transactions.groupBy('mcc','mcc_description').agg(F.count('id').alias('merchant_total_transactions'))
merchant_category_fraud=transactions.filter(F.col('Fraud_Label')==True).groupBy('mcc').agg(F.count('id').alias('merchant_fraud_transactions'))
merchant_category_fraud_rate=merchant_category_total.join(merchant_category_fraud,on='mcc',how='left').withColumn('merchant_fraud_transactions',F.when(F.col('merchant_fraud_transactions').isNull(),0).otherwise(F.col('merchant_fraud_transactions'))).withColumn('rate',F.col('merchant_fraud_transactions')/F.col('merchant_total_transactions')).orderBy(F.desc('rate')).dropDuplicates()
merchant_category_fraud_rate.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.merchant_category_fraud_rate')
display(merchant_category_fraud_rate)

mcc,mcc_description,merchant_total_transactions,merchant_fraud_transactions,rate
4411,Cruise Lines,276,165,0.5978260869565217
5733,Music Stores - Musical Instruments,204,76,0.37254901960784315
3006,Miscellaneous Fabricated Metal Products,245,29,0.11836734693877551
5045,"Computers, Computer Peripheral Equipment",1883,204,0.10833775889537971
3144,Floor Covering Stores,222,23,0.1036036036036036
3005,Miscellaneous Metal Fabrication,256,22,0.0859375
5732,Electronics Stores,4689,402,0.08573256557901472
3009,Fabricated Structural Metal Products,273,22,0.08058608058608059
5094,Precious Stones and Metals,3525,242,0.06865248226950355
5712,"Furniture, Home Furnishings, and Equipment Stores",2600,170,0.06538461538461539


### Question 6:
Are there specific merchants with unusually high fraud volume?

In [0]:
mcc_count_fraud=transactions.filter(F.col('Fraud_Label')==True).groupBy('mcc').agg(F.count('id').alias('count_fraud')).join(transactions.select('mcc','mcc_description'),on='mcc',how='inner').dropDuplicates().orderBy(F.desc('count_fraud'))
mcc_count_fraud.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.category_count_fraud')
display(mcc_count_fraud)

mcc,count_fraud,mcc_description
5311,2251,Department Stores
5300,991,Wholesale Clubs
5310,859,Discount Stores
4829,725,Money Transfer
5912,479,Drug Stores and Pharmacies
5815,449,"Digital Goods - Media, Books, Apps"
5411,425,"Grocery Stores, Supermarkets"
5732,402,Electronics Stores
5651,385,Family Clothing Stores
5719,313,Miscellaneous Home Furnishing Stores


### Question 7:
How does fraud distribution vary by time of day (morning vs night)?    
- Analysis is made with morning being 5am - 11:59am and night being 8pm - 4:59am

In [0]:
morning=transactions.filter((F.hour(F.col('date')).between(5,11)) &(F.col('Fraud_Label')==True)).agg(F.count('id').alias('morning_fraud')).withColumn('ref_id',F.lit(1))
night=transactions.filter((F.hour("date").isin(20, 21, 22, 23, 0, 1, 2, 3, 4)) &(F.col('Fraud_Label')==True)).agg(F.count('id').alias('night_fraud')).withColumn('ref_id',F.lit(1))
morn_vs_night=morning.join(night,on='ref_id',how='inner').drop('ref_id')
morn_vs_night.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.morn_vs_night')
display(morn_vs_night)

morning_fraud,night_fraud
5741,458


### Question 8:
What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
fraud_avg=transactions.filter(F.col('Fraud_Label')==True).agg(F.avg('amount').alias('avg_fraud'))
nonfraud_avg=transactions.filter(F.col('Fraud_Label')==False).agg(F.avg('amount').alias('avg_nonfraud'))
print(f"Average Fraud amount: {fraud_avg.collect()[0][0]}")
print(f"Average Non-Fraud Amount: {nonfraud_avg.collect()[0][0]}")

Average Fraud amount: 110.234682
Average Non-Fraud Amount: 42.848614


### Question 9:
Which merchant category has the highest total fraud amount?

In [0]:
mcc_amount_fraud=transactions.filter(F.col('Fraud_Label')==True).groupBy('mcc').agg(F.sum('amount').alias('amount_fraud')).join(transactions.select('mcc','mcc_description'),on='mcc',how='inner').dropDuplicates().orderBy(F.desc('amount_fraud'))
mcc_amount_fraud.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.mcc_amount_fraud')
display(mcc_amount_fraud)

mcc,amount_fraud,mcc_description
5311,225647.19,Department Stores
4411,185946.78,Cruise Lines
5300,113827.65,Wholesale Clubs
5310,81214.89,Discount Stores
4829,66101.52,Money Transfer
5732,61171.38,Electronics Stores
5712,56989.45,"Furniture, Home Furnishings, and Equipment Stores"
5719,34238.45,Miscellaneous Home Furnishing Stores
4814,33625.04,Telecommunication Services
5651,30051.56,Family Clothing Stores


### Question 10:
What are the total monetary losses due to fraud each day?

In [0]:
total_daily_losses=transactions.filter((F.col('Fraud_Label')==True)&(F.col('amount')>0)).groupBy(F.date_format('date','yyyy-MM-dd').alias('date')).agg(F.sum('amount').alias('amount_fraud')).orderBy(F.desc('amount_fraud'))
total_daily_losses.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.total_daily_losses')
display(total_daily_losses)

date,amount_fraud
2010-11-14,7266.59
2010-02-19,6986.25
2013-09-28,6611.12
2010-03-26,6491.05
2013-10-19,6427.63
2010-04-11,5656.76
2015-12-26,5605.53
2016-02-07,5553.70
2015-12-21,5142.06
2010-03-22,5089.06


### Question 11:
How many unique users commit fraudulent transactions per week?

In [0]:
user_count=transactions.filter(F.col('Fraud_Label')==True).withColumn('Year-Week',F.concat_ws('-',F.year(F.col('date')),F.weekofyear(F.col('date')))).groupBy('Year-Week').agg(F.count('client_id').alias('user_count')).dropDuplicates().orderBy('user_count',ascending=False)
user_count.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.user_count')
display(user_count)

Year-Week,user_count
2013-42,120
2015-52,110
2015-49,109
2013-39,104
2015-51,94
2015-28,89
2010-21,89
2010-18,84
2016-1,78
2010-12,77


### Question 12:
Do fraud patterns show seasonal or monthly spikes?

In [0]:
seasons_months=transactions.filter(F.col('Fraud_Label')==True).withColumn('Month',F.month(F.col('date'))).withColumn('Season',F.when(F.col('Month').isin(12,1,2), 'Winter').when(F.col('Month').isin(3,4,5), 'Spring').when(F.col('Month').isin(6,7,8), 'Summer').otherwise('Fall')).withColumn('Month',F.when(F.col('Month')==1,'January').when(F.col('Month')==2,'February').when(F.col('Month')==3,'March').when(F.col('Month')==4,'April').when(F.col('Month')==5,'May').when(F.col('Month')==6,'June').when(F.col('Month')==7,'July').when(F.col('Month')==8,'August').when(F.col('Month')==9,'September').when(F.col('Month')==10,'October').when(F.col('Month')==11,'November').when(F.col('Month')==12,'December'))
seasons=seasons_months.groupBy('Season').agg(F.count('id').alias('fraud_count')).dropDuplicates()
months=seasons_months.groupBy('Month').agg(F.count('id').alias('fraud_count')).dropDuplicates().orderBy('Month')
months.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.months')
seasons.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.seasons')
display(seasons)
display(months)

                                                                            

Season,fraud_count
Summer,3318
Fall,3345
Spring,3435
Winter,3234


Month,fraud_count
April,1171
August,1328
December,1201
February,1030
January,1003
July,1118
June,872
March,1167
May,1097
November,1089


### Question 13:
How has user behavior changed before versus after a fraudulent event?    
Lets look in terms of RFM model

In [0]:
from pyspark.sql.window import Window

window=Window.partitionBy('client_id')
first_fraud_date=F.min(F.when(F.col('Fraud_Label')==True, F.col('date'))).over(window)
user_RFM = transactions.join(users, transactions.client_id==users.id, how='inner').orderBy('client_id').withColumn('first_fraud_date', first_fraud_date).filter(F.col('Fraud_Label')==False).withColumn('period', F.when(F.col('date') < F.col('first_fraud_date'), 'before_fraud').when(F.col('date') > F.col('first_fraud_date'), 'after_fraud'))
user_RFM=user_RFM.groupBy('client_id','period').agg(F.count(F.col('transactions.id')).alias('Frequency'), F.sum('amount').alias('Monetary'), F.when(F.col('period')=='after_fraud', F.min('date')).when(F.col('period')=='before_fraud', F.max('date')).alias('Recency')).dropDuplicates().orderBy('client_id','period')
display(user_RFM)

client_id,period,Frequency,Monetary,Recency
0,after_fraud,3513,177228.52,2015-10-31T08:59:00.000Z
0,before_fraud,5127,248532.30,2015-10-28T14:01:00.000Z
1,after_fraud,1873,66093.24,2016-11-21T09:43:00.000Z
1,before_fraud,4878,158011.73,2016-11-20T14:24:00.000Z
2,after_fraud,2281,60525.32,2016-09-15T11:47:00.000Z
2,before_fraud,4778,132319.25,2016-09-12T21:30:00.000Z
3,after_fraud,2559,122169.18,2013-10-18T14:32:00.000Z
3,before_fraud,1441,63906.25,2013-10-17T14:23:00.000Z
4,after_fraud,9658,383025.39,2010-05-09T08:52:00.000Z
4,before_fraud,347,13811.95,2010-05-07T21:10:00.000Z


In [0]:
before=user_RFM.filter(F.col('period')=='before_fraud').withColumnRenamed('Frequency','Frequency_before').withColumnRenamed('Monetary','Monetary_before').withColumnRenamed('Recency','Recency_before').drop('period')
after=user_RFM.filter(F.col('period')=='after_fraud').withColumnRenamed('Frequency','Frequency_after').withColumnRenamed('Monetary','Monetary_after').withColumnRenamed('Recency','Recency_after').drop('period')
user_behaviour=before.join(after,on='client_id',how='inner')
user_behaviour.write.mode('overwrite').saveAsTable('databricks_fundamentals.gold.user_behaviour')
display(user_behaviour)


client_id,Frequency_before,Monetary_before,Recency_before,Frequency_after,Monetary_after,Recency_after
63,2069,47019.31,2012-02-09T10:27:00.000Z,8452,167605.48,2012-02-09T15:23:00.000Z
48,5122,167248.10,2019-07-18T06:54:00.000Z,161,5806.78,2019-07-20T16:03:00.000Z
184,1620,71682.27,2013-10-24T13:14:00.000Z,2564,112649.61,2013-10-25T14:14:00.000Z
155,11,181.76,2010-01-22T14:31:00.000Z,5584,116138.87,2010-01-29T22:55:00.000Z
198,1779,101164.45,2015-05-08T16:00:00.000Z,1508,93776.09,2015-05-12T00:01:00.000Z
145,2939,64202.62,2016-09-23T12:29:00.000Z,1517,29024.76,2016-09-25T13:19:00.000Z
142,3626,228443.89,2014-03-07T11:38:00.000Z,4845,295207.40,2014-03-08T07:55:00.000Z
211,523,9069.30,2010-08-11T15:35:00.000Z,7886,138790.89,2010-08-12T12:22:00.000Z
157,4672,187468.27,2015-12-08T10:24:00.000Z,3078,120561.10,2015-12-09T15:33:00.000Z
51,3010,144584.43,2015-09-19T15:50:00.000Z,2541,116851.66,2015-09-20T11:09:00.000Z


### Question 14:
Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
high_value=transactions.filter((F.col('amount')>300)&(F.col('Fraud_Label')==True)).agg(F.count('id').alias('high_value_fraud')).withColumn('ref_id',F.lit(1))
low_value=transactions.filter((F.col('amount')<=300)&(F.col('amount')>0)&(F.col('Fraud_Label')==True)).agg(F.count('id').alias('low_value_fraud')).withColumn('ref_id',F.lit(1))
low_vs_high=low_value.join(high_value,on='ref_id',how='inner').drop('ref_id')
display(low_vs_high)

low_value_fraud,high_value_fraud
11644,1193
